In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import builtins
from google.colab import drive

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
drive.mount('/content/drive')

DRIVE_DATA_PATH = "/content/drive/My Drive/WVS_happiness_study"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python3 --version

Python 3.12.12


In [6]:
# Added for split notebook reruns: load Data Cleaning output if this notebook is run independently.
from pathlib import Path

if "df_cleaned" not in globals():
    _cleaned_candidates = [
        Path(f"{DRIVE_DATA_PATH}/cleaned_df.csv"),
        Path(f"{DRIVE_DATA_PATH}data/cleaned_df.csv"),
        Path(f"{DRIVE_DATA_PATH}data/cleaned_df.csv.zip"),
        Path(f"{DRIVE_DATA_PATH}/data/processed/cleaned_df.csv"),
        Path(f"{DRIVE_DATA_PATH}/data/processed/cleaned_df.csv.zip")
    ]
    for _path in _cleaned_candidates:
        if _path.exists():
            df_cleaned = pd.read_csv(_path)
            print(f"Loaded df_cleaned from {_path}")
            break
    else:
        raise FileNotFoundError("Could not find cleaned_df.csv. Run the Data Cleaning notebook first.")


Loaded df_cleaned from /content/drive/My Drive/WVS_happiness_study/data/processed/cleaned_df.csv


In [7]:
# Added for split notebook reruns: load questionnaire metadata if needed.
from pathlib import Path

if "variable_table" not in globals():
    _questionnaire_candidates = [
        Path(f"{DRIVE_DATA_PATH}/Questionnaire.xlsx"),
        Path(f"{DRIVE_DATA_PATH}/data/Questionnaire.xlsx"),
        Path(f"{DRIVE_DATA_PATH}/data/raw/Questionnaire.xlsx")
    ]
    for _path in _questionnaire_candidates:
        if _path.exists():
            variable_table = pd.read_excel(_path)
            print(f"Loaded variable_table from {_path}")
            break
    else:
        raise FileNotFoundError("Could not find Questionnaire.xlsx.")


Loaded variable_table from /content/drive/My Drive/WVS_happiness_study/data/raw/Questionnaire.xlsx


In [8]:
def var_description(df, scale_description=False, new_var_name=False):
    for col in df.columns:
        print(col, f'dtype={df[col].dtype}')
        print('Unique values:', df[col].sort_values(ascending=True).unique().tolist())
        if scale_description:

            if new_var_name:
                cond = variable_table['New Variable Name'] == col
            else:
                cond = variable_table['Q_num'] == col
            scale_desc = variable_table[cond]['Q_Scale']

            if not scale_desc.empty:
                # Clean the scale description by replacing \n with spaces
                cleaned_desc = scale_desc.iloc[0].replace('\n', ' ')
                # Optional: Remove extra spaces that might result from the replacement
                cleaned_desc = ' '.join(cleaned_desc.split())
                print('Scale description:', cleaned_desc)
            else:
                print('Scale description: Not found')
        print("-" * 100)

#show and retrieve the missing data percentage by threshold
def get_na_cols_perc(df, threshold):
    na_cols = []
    count = 0
    for col in df.columns:
        na_percentage = df[col].isna().sum()/ len(df)
        if(na_percentage >= threshold):
            print(col, f": {na_percentage}")
            na_cols.append(col)
            count += 1
    print("-"*100)
    print("Total column: ", count)
    return na_cols


### Feature Engineering

#### Import Packages

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
import json
from joblib import Parallel, delayed
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")


In [10]:
df_fea_eng = df_cleaned
df_fea_eng.head()

,country,belonging1,belonging2,esteem1,civic1,esteem2,belonging3,trust1,growth1,growth2,civic2,growth3,ethical1,ethical2,growth4,growth5,ethical3,ethical4,ethical5,ethical6,ethical7,ethical8,ethical9,ethical10,ethical11,ethical12,ethical13,esteem3,belonging4,ga_leaders_male_better,ga_edu_male_priority,ga_exec_male_better,esteem4,ga_jobs_male_priority,emp_prior_immig,ga_income_woman_problem,ethical14,civic3,civic4,ethical15,civic5,civic6,society_concern,growth6,growth7,growth8,happy,physio1,esteem5,sact,safety1,physio2,safety2,physio3,safety3,safety4,esteem6,trust2,trust3,trust4,trust5,trust6,trust7,trust8,trust9,trust10,trust11,trust12,trust13,trust14,trust15,trust16,trust17,trust18,trust19,trust20,trust21,trust22,trust23,trust24,trust25,trust26,trust27,trust28,trust29,trust30,trust31,trust32,trust33,trust34,ethical16,UN_country,IMF_head,Amn_Int_org_prob,civic7,civic8,civic9,civic10,civic11,civic12,civic13,civic14,civic15,civic16,civic17,civic18,income_eq_vs_diff,pri_vs_state,gov_vs_individual,comp_good_vs_harm,growth9,protect_env_vs_econ,ethical17,ethical18,ethical19,ethical20,ethical21,ethical22,ethical23,ga_women_less_corrupt,ethical24,compassion1,civic19,compassion2,ethical25,compassion3,ethical26,compassion4,civic20,ethical27,immig_policy_prefer,safety5,safety6,safety7,safety8,safety9,safety10,safety11,safety12,safety13,safety14,safety15,safety16,safety17,safety18,safety19,safety20,safety21,safety22,freedom_eq,freedom_sec,civic21,country_aim1,country_aim2,respon_aim1,respon_aim2,important_choice1,important_choice2,growth10,civic22,ethical28,ethical29,growth11,ethical30,growth12,ethical31,growth13,ethical32,growth14,religious1,ethical33,trust35,growth15,religious2,religious3,religious4,ethical34,ethical35,ethical36,ethical37,ethical38,ethical39,ethical40,ethical41,ethical42,ethical43,ethical44,ethical45,ethical46,ethical47,ethical48,ethical49,ethical50,ethical51,ethical52,ethical53,ethical54,ethical55,ethical56,civic23,civic24,civic25,civic26,civic27,civic28,civic29,civic30,civic31,civic32,civic33,civic34,civic35,civic36,civic37,civic38,civic39,civic40,civic41,civic42,civic43,civic44,civic45,civic46,party_vote,ethical57,ethical58,ethical59,ethical60,ethical61,ethical62,ethical63,ethical64,ethical65,ethical66,civic47,ethical67,ethical68,ethical69,ethical70,civic48,ethical71,l_vs_r_political,ethical72,ethical73,civic49,ethical74,ethical75,ethical76,ethical77,ethical78,ethical79,civic50,civic51,civic52,ethical80,civic53,growth16,growth17,growth18,growth19,growth20,sex,date_of_birth,age,immigrant,mom_immigrant,dad_immigrant,birth_country,mom_birth_country,dad_birth_country,citizenship,num_of_household,live_with_parent,mother_tongue,marital_status,num_of_child,education_level,spouse_education_level,mom_education_level,dad_education_level,employment_status,spouse_employment_status,occupational_group,spouse_occupational_group,dad_occupational_group,Employment_sector,chief_wage,family_saving,social_class,income_scale,religiosity,race
0,CYP,4.0,4.0,4.0,1.0,4.0,4.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,4.0,1.0,2.0,2.0,2.0,1.0,2.0,5.0,4.0,1.0,3.0,5.0,3.0,2.0,3.0,2.0,3.0,1.0,NaN,3.0,2.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,3.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,3.0,1.0,2.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,1.0,1.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,3.0,3.0,2.0,2.0,0.0,1.0,3.0,3.0,4.0,1.0,3.0,3.0,1.0,5.0,5.0,1.0,3.0,5.0,1.0,NaN,NaN,NaN,NaN,3.0,1.0,5.0,1.0,0.0,0.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,3.0,3.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0,4.0,4.0,1.0,1.0,1.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,NaN,1.0,NaN,1.0,2.0,2.0,1.0,2.0,NaN,2.0,3.0,1.0,4.0,4.0,1.0,1.0,4.0,1.0,NaN,5.0,3.0,5.0,5.0,1.0,5.0,5.0,5.0,5.0,5.0,1.0,1.

In [11]:
df_fea_eng.shape

(95528, 292)

#### Drop Redundant Features

There are several features to be removed which may introduce noise or bias:
- Remove country variable.
- Remove race.
- Drop the questions about UN Security Council countries, IMF headquarters, and Amnesty International which focus on factual knowledge of international organizations rather than personal experiences, socioeconomic status, or well-being.
- Remove birth country for respondent and parents.
- Remove date of birth which is redundant with age.
- Party vote is not suitable to include in modelling.
- mother tongue is not ideal to include in predictive modelling.
- Drop those variables with spouse related because there are relied on marital status whereby a respondent with single status won't have spouse.

In [12]:
cols_to_ex = ['date_of_birth', 'party_vote', 'mother_tongue', 'country', 'race']
org_knowledge_cols = ['UN_country', 'IMF_head', 'Amn_Int_org_prob']
birth_cols = [c for c in df_fea_eng.columns if 'birth' in c.lower()]
spouse_cols = [c for c in df_fea_eng.columns if c.startswith('spouse_')]

cols_to_drop = cols_to_ex + spouse_cols + birth_cols + org_knowledge_cols
df_fea_eng = df_fea_eng.drop(columns=cols_to_drop)

In [13]:
df_fea_eng.shape

(95528, 278)

#### VIF

In [ ]:
def get_grouped_cols(cols_grp_name, exclude=None):
    """
    Return columns which contain any of the strings in cols_grp_name,
    optionally excluding columns that contain strings in 'exclude',
    but only if the exclude string is longer than the group name.

    cols_grp_name: str or list of str
    exclude: list of str
    """
    if exclude is None:
        exclude = []

    # Ensure cols_grp_name is a list
    if isinstance(cols_grp_name, str):
        cols_grp_name = [cols_grp_name]

    cols = []
    for col in df_fea_eng.columns:
        # Check if any of the group names are in the column
        if any(g in col for g in cols_grp_name):
            # Check exclude list
            to_exclude = False
            for e in exclude:
                if any(len(e) > len(g) and e in col for g in cols_grp_name):
                    to_exclude = True
                    break
            if not to_exclude:
                cols.append(col)
    return cols


Compare mean and median using simple impute to decide which to be used for each ordinal variable.

In [ ]:
def choose_mean_or_median(df, cols, mask_fraction=0.1, random_state=42):
    np.random.seed(random_state)

    total_mae_mean = 0
    total_mae_median = 0
    count = 0

    for col in cols:
        s = df[col].copy()
        if s.isna().all():
            continue

        mask = s.notna() & (np.random.rand(len(s)) < mask_fraction)
        if mask.sum() == 0:
            continue

        true_vals = s[mask].copy()
        s_masked = s.copy()
        s_masked[mask] = np.nan

        # Mean imputation
        mean_imp = SimpleImputer(strategy='mean').fit_transform(
            s_masked.values.reshape(-1,1)
        ).flatten()
        total_mae_mean += np.mean(np.abs(mean_imp[mask] - true_vals))

        # Median imputation
        median_imp = SimpleImputer(strategy='median').fit_transform(
            s_masked.values.reshape(-1,1)
        ).flatten()
        total_mae_median += np.mean(np.abs(median_imp[mask] - true_vals))

        count += 1

    # Final decision
    final_choice = "median" if total_mae_median < total_mae_mean else "mean"

    return {
        "final_choice": final_choice,
        "avg_mae_mean": total_mae_mean / count if count else None,
        "avg_mae_median": total_mae_median / count if count else None,
        "variables_used": count
    }


Exclude nominal data for imputing.

In [ ]:
# exclude the nominal data
filtered_vars = variable_table[~variable_table['Type'].isin(['Nominal'])]
cols_to_imp = [col for col in filtered_vars['New Variable Name'] if col in df_fea_eng.columns]

df_vif = df_fea_eng[cols_to_imp].copy()

In [ ]:
df_vif.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95528 entries, 0 to 95527
Columns: 261 entries, belonging1 to income_scale
dtypes: Int64(261)
memory usage: 214.0 MB


In [ ]:
# If range is not callable, restore it
if not callable(range):
    print("Restoring range function...")
    range = builtins.range
    len = builtins.len

def inter_vif(df, cols, target, threshold=10.0):

    df_use = df[cols].dropna().copy()
    temp_vars = cols.copy()

    kept_vars = []      # store HIGH VIF variables (VIF > 10)
    dropped = []        # store LOW VIF variables (≤ 10)
    no_multi_grp_vars = []  # store variables from group with no high-VIF

    print(f"\n🌟 Processing group with {len(cols)} variables:", cols)

    pbar = tqdm(desc=f"VIF loop ({cols[0].split('_')[0]})",
                total=len(cols), leave=False)

    while True:
        X = df_use[temp_vars].astype(np.float32).values

        # Compute VIF for all remaining vars
        vif_list = Parallel(n_jobs=-1)(
            delayed(variance_inflation_factor)(X, i)
            for i in range(X.shape[1])
        )

        vif_df = pd.DataFrame({"Feature": temp_vars, "VIF": vif_list})
        vif_df = vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)

        max_vif = vif_df.iloc[0]["VIF"]
        worst = vif_df.iloc[0]["Feature"]

        # If highest VIF is HIGH → KEEP it
        if max_vif > threshold:
            kept_vars.append(worst)
            print(f"  → Added {worst} (VIF={max_vif:.2f})")

            temp_vars.remove(worst)     # remove from pool
            pbar.update(1)

        else:
            # All remaining variables are LOW VIF → drop them all
            for feat, v in zip(vif_df["Feature"], vif_df["VIF"]):
                dropped.append((feat, v))
                print(f"  → Dropped {feat} (VIF={v:.2f})")

            # If no high-VIF variable found at all, mark all as no_multicollinearity
            if len(kept_vars) == 0:
                no_multi_grp_vars.extend(temp_vars)
            break

        # Stop if only 1 variable left (no more VIF possible)
        if len(temp_vars) <= 1:
            break

    pbar.close()

    # Determine the winner among high-VIF variables (best correlation with target)
    if len(kept_vars) > 1:
        corrs = df[kept_vars].corrwith(df[target]).abs()
        winner = corrs.idxmax()
        winner_corr = corrs[winner]
    elif len(kept_vars) == 1:
        winner = kept_vars[0]
        winner_corr = df[winner].corr(df[target])
    else:
        winner = None
        winner_corr = None

    survivors = kept_vars  # all high-VIF variables

    if winner is not None:
        print(f"\nWinner: {winner} (|corr| = {winner_corr:.4f})")
    else:
        print("\nWinner: None")


    if no_multi_grp_vars:
        print(f"No-multicollinearity variables: {no_multi_grp_vars}")
    else:
        print(f"Survivors (high-VIF): {survivors}")

    return winner, survivors, no_multi_grp_vars, dropped


def intra_vif(df, cols, threshold=10.0):
    """
    Compute VIF for a group of variables (intra-group multicollinearity).

    Args:
        df (pd.DataFrame): DataFrame containing the variables.
        cols (list): List of column names to check.
        threshold (float): VIF threshold above which a variable is considered highly collinear.

    Returns:
        kept_vars (list): Variables kept (VIF ≤ threshold or manually kept).
        dropped (list): Variables dropped due to high VIF (> threshold).
    """

    df_use = df[cols].dropna().copy()
    temp_vars = cols.copy()

    kept_vars = []   # variables to keep
    dropped = []     # variables to drop due to high VIF

    print(f"\n🌟 Processing intra-VIF for {len(cols)} variables:", cols)

    pbar = tqdm(total=len(cols), desc="Intra-VIF loop", leave=False)

    while True:
        X = df_use[temp_vars].astype(np.float32).values

        # Compute VIF for all remaining variables
        vif_list = Parallel(n_jobs=-1)(
            delayed(variance_inflation_factor)(X, i) for i in range(X.shape[1])
        )

        vif_df = pd.DataFrame({"Feature": temp_vars, "VIF": vif_list})
        vif_df = vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)

        max_vif = vif_df.iloc[0]["VIF"]
        worst = vif_df.iloc[0]["Feature"]

        if max_vif > threshold:
            # Drop the variable with highest VIF
            dropped.append(worst)
            temp_vars.remove(worst)
            print(f"  → Dropped {worst} (VIF={max_vif:.2f})")
        else:
            # All remaining variables are below threshold → keep them
            kept_vars.extend(temp_vars)
            break

        if len(temp_vars) <= 1:
            kept_vars.extend(temp_vars)
            break

        pbar.update(1)

    pbar.close()

    print(f"\nRemaining variables after intra-VIF: {kept_vars}")

    return kept_vars, dropped


Restoring range function...


In [ ]:
fractions = [0.10, 0.15, 0.20]
all_results = {}

for frac in fractions:
    results = choose_mean_or_median(df_vif, df_vif.columns, mask_fraction=frac)
    all_results[frac] = results

In [ ]:
for frac in fractions:
    print(frac, "→", all_results[frac]['final_choice'])


0.1 → median
0.15 → median
0.2 → median


Retrieve median and mean columns.

In [ ]:
# Imputer for median columns
imp_median = SimpleImputer(strategy='median')
df_median = pd.DataFrame(imp_median.fit_transform(df_vif), columns=df_vif.columns)

In [ ]:
df_imputed = df_median.copy().astype(int)
df_imputed.tail()

,belonging1,belonging2,esteem1,civic1,esteem2,belonging3,trust1,growth1,growth2,civic2,growth3,ethical1,ethical2,growth4,growth5,ethical3,ethical4,ethical5,ethical6,ethical7,ethical8,ethical9,ethical10,ethical11,ethical12,ethical13,esteem3,belonging4,ga_leaders_male_better,ga_edu_male_priority,ga_exec_male_better,esteem4,ga_jobs_male_priority,emp_prior_immig,ga_income_woman_problem,ethical14,civic3,civic4,ethical15,civic5,civic6,growth6,growth7,growth8,happy,physio1,esteem5,sact,safety1,physio2,safety2,physio3,safety3,safety4,esteem6,trust2,trust3,trust4,trust5,trust6,trust7,trust8,trust9,trust10,trust11,trust12,trust13,trust14,trust15,trust16,trust17,trust18,trust19,trust20,trust21,trust22,trust23,trust24,trust25,trust26,trust27,trust28,trust29,trust30,trust31,trust32,trust33,trust34,ethical16,civic7,civic8,civic9,civic10,civic11,civic12,civic13,civic14,civic15,civic16,civic17,civic18,income_eq_vs_diff,pri_vs_state,gov_vs_individual,comp_good_vs_harm,growth9,ethical17,ethical18,ethical19,ethical20,ethical21,ethical22,ethical23,ga_women_less_corrupt,ethical24,compassion1,civic19,compassion2,ethical25,compassion3,ethical26,compassion4,civic20,ethical27,safety5,safety6,safety7,safety8,safety9,safety10,safety11,safety12,safety13,safety14,safety15,safety16,safety17,safety18,safety19,safety20,safety21,safety22,civic21,growth10,civic22,ethical28,ethical29,growth11,ethical30,growth12,ethical31,growth13,ethical32,growth14,religious1,ethical33,trust35,growth15,religious2,religious3,religious4,ethical34,ethical35,ethical36,ethical37,ethical38,ethical39,ethical40,ethical41,ethical42,ethical43,ethical44,ethical45,ethical46,ethical47,ethical48,ethical49,ethical50,ethical51,ethical52,ethical53,ethical54,ethical55,ethical56,civic23,civic24,civic25,civic26,civic27,civic28,civic29,civic30,civic31,civic32,civic33,civic34,civic35,civic36,civic37,civic38,civic39,civic40,civic41,civic42,civic43,civic44,civic45,civic46,ethical57,ethical58,ethical59,ethical60,ethical61,ethical62,ethical63,ethical64,ethical65,ethical66,ethical67,civic47,ethical68,ethical69,ethical70,civic48,ethical71,l_vs_r_political,ethical72,ethical73,civic49,ethical74,ethical75,ethical76,ethical77,ethical78,ethical79,civic50,civic51,civic52,ethical80,civic53,growth16,growth17,growth18,growth19,growth20,sex,age,immigrant,mom_immigrant,dad_immigrant,citizenship,num_of_household,live_with_parent,marital_status,num_of_child,education_level,mom_education_level,dad_education_level,chief_wage,social_class,income_scale
95523,4,3,3,3,4,4,1,1,1,0,0,1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,3,4,1,1,1,4,1,5,2,1,5,5,3,5,4,1,1,3,3,3,3,3,3,4,4,2,4,4,3,1,4,3,3,3,3,3,4,4,1,1,1,2,2,1,1,1,1,1,2,1,1,1,3,3,1,2,3,2,2,1,3,1,2,2,1,2,0,0,0,2,2,0,2,0,2,3,3,3,3,4,5,2,2,2,2,2,4,2,3,3,2,2,0,2,0,2,0,0,2,2,2,4,3,3,3,3,1,1,1,4,3,1,1,4,4,2,1,3,3,5,5,3,3,5,1,1,1,1,4,3,5,5,1,0,0,1,5,5,5,5,5,3,5,5,3,5,5,5,5,3,5,5,5,5,3,4,2,1,4,3,5,5,5,2,5,2,1,5,3,3,3,3,3,3,3,3,1,3,1,1,4,4,4,4,1,2,2,4,4,3,4,2,4,3,1,1,1,4,1,5,5,5,5,5,1,3,2,5,5,5,4,1,4,4,2,4,4,2,2,0,51,0,0,0,1,4,0,1,2,5,3,3,1,3,5
95524,4,4,3,3,4,4,0,1,0,0,0,1,0,1,1,1,0,1,0,1,0,1,0,1,0,0,3,3,1,1,1,4,1,1,1,2,3,4,3,4,2,2,3,3,4,3,4,5,5,4,4,2,4,4,1,2,4,3,3,3,3,3,3,3,2,2,3,3,3,2,2,2,3,3,2,2,2,3,3,3,2,3,2,2,3,2,3,2,3,2,0,0,0,0,0,0,0,0,0,0,0,3,3,3,2,4,4,3,3,3,3,2,3,2,4,4,2,2,2,2,2,2,2,2,4,4,2,4,4,2,4,3,1,1,1,2,1,1,1,2,2,2,1,4,4,2,1,5,4,5,1,1,1,1,4,4,5,5,1,0,0,5,5,5,5,5,5,5,5,5,3,5,5,5,5,5,5,5,5,5,4,3,3,2,3,3,3,5,5,5,4,5,5,4,3,2,3,3,3,3,3,3,3,3,2,1,4,4,4,4,3,4,3,4,4,4,4,4,4,4,1,1,1,4,1,4,4,2,5,4,1,5,2,4,5,5,4,2,2,2,4,3,3,2,2,0,35,0,0,0,1,4,0,1,2,5,3,3,1,3,3
95525,4,3,3,3,4,2,1,0,0,1,1,1,0,1,0,0,0,0,0,1,0,1,0,0,0,0,3,4,4,1,3,3,1,4,2,3,2,1,5,5,2,3,3,3,3,4,4,4,3,4,4,4,4,4,2,1,4,3,4,2,3,3,1,3,1,2,2,2,1,1,1,1,2,3,2,2,3,2,2,2,1,2,2,2,1,3,3,3,1,1,2,0,2,0,0,0,0,0,0,0,0,3,5,5,2,5,5,2,3,3,3,2,3,2,1,3,1,2,0,2,0,2,0,0,3,2,3,4,3,3,3,3,1,1,1,2,3,1,1,2,2,2,1,5,2,1,1,4,5,1,1,0,1,1,1,2,1,1,1,0,0,2,5,3,5,4,4,3,3,3,3,1,5,3,5,2,2,5,3,3,2,4,1,2,3,2,2,5,5,5,1,5,5,5,3,3,2,2,2,2,2,3,3,3,2,1,4,4,3,4,2,2,1,4,4,2,3,

In [ ]:
target = 'happy'
# Your grouped columns (apply in inter-VIF)
groups = {
    "belonging": get_grouped_cols('belonging'),
    "esteem": get_grouped_cols('esteem'),
    "safety": get_grouped_cols('safety'),
    "physio": get_grouped_cols('physio'),
    "civic": get_grouped_cols('civic', exclude=['civic_']),
    "trust" : get_grouped_cols('trust'),
    "growth": get_grouped_cols('growth'),
    "ethical": get_grouped_cols('ethical', exclude=['ethical_']),
    "compassion": get_grouped_cols('compassion'),
    "education": get_grouped_cols('education'),
    "socioeconomic": get_grouped_cols(['social_class','income_scale', 'chief_wage']),
    "immigrant": get_grouped_cols('immigrant'),
    "religious": get_grouped_cols('religious'),
    "gender_attitudes": get_grouped_cols('ga_')
}

# non-grouped mainly for intra-VIF
flattened_grp_list = [item for sublist in groups.values() for item in sublist] # get all grouped columns in list form
non_grp_vars = df_imputed.select_dtypes(exclude='category').columns.difference(flattened_grp_list).to_list()
non_grp_vars = [c for c in non_grp_vars if c not in target] # exclude target (happy)

print("Grouped variables:\n", groups)
print("Ungrouped variables:\n", non_grp_vars)

Grouped variables:
 {'belonging': ['belonging1', 'belonging2', 'belonging3', 'belonging4'], 'esteem': ['esteem1', 'esteem2', 'esteem3', 'esteem4', 'esteem5', 'esteem6'], 'safety': ['safety1', 'safety2', 'safety3', 'safety4', 'safety5', 'safety6', 'safety7', 'safety8', 'safety9', 'safety10', 'safety11', 'safety12', 'safety13', 'safety14', 'safety15', 'safety16', 'safety17', 'safety18', 'safety19', 'safety20', 'safety21', 'safety22'], 'physio': ['physio1', 'physio2', 'physio3'], 'civic': ['civic1', 'civic2', 'civic3', 'civic4', 'civic5', 'civic6', 'civic7', 'civic8', 'civic9', 'civic10', 'civic11', 'civic12', 'civic13', 'civic14', 'civic15', 'civic16', 'civic17', 'civic18', 'civic19', 'civic20', 'civic21', 'civic22', 'civic23', 'civic24', 'civic25', 'civic26', 'civic27', 'civic28', 'civic29', 'civic30', 'civic31', 'civic32', 'civic33', 'civic34', 'civic35', 'civic36', 'civic37', 'civic38', 'civic39', 'civic40', 'civic41', 'civic42', 'civic43', 'civic44', 'civic45', 'civic46', 'civic47', 

In [ ]:
dropped_vars = {}
winner_grp_var = {}
survived_vars = {}
non_multi_corr_grp_vars = {} # track group variables with no high VIF
for name, cols in groups.items():
    print(f"\n{'='*8} Processing category: {name} {'='*8}")

    winner, kept_vars, non_multi_corr_vars, dropped = inter_vif(
        df_imputed,
        cols,
        target='happy',
        threshold=10
    )

    winner_grp_var[name] = winner
    dropped_vars[name] = dropped
    survived_vars[name] = kept_vars  # kept vars for questionnaires from same group

    if non_multi_corr_vars:
        non_multi_corr_grp_vars[name] = non_multi_corr_vars


print("\nFinal inter-VIF winners:", winner_grp_var)




======== Processing category: belonging ========

🌟 Processing group with 4 variables: ['belonging1', 'belonging2', 'belonging3', 'belonging4']


VIF loop (belonging1):  25%|██▌       | 1/4 [00:02<00:06,  2.28s/it]

  → Added belonging1 (VIF=35.15)


  → Added belonging2 (VIF=11.01)
  → Dropped belonging4 (VIF=3.97)
  → Dropped belonging3 (VIF=3.97)

Winner: belonging1 (|corr| = 0.1059)
Survivors (high-VIF): ['belonging1', 'belonging2']

======== Processing category: esteem ========

🌟 Processing group with 6 variables: ['esteem1', 'esteem2', 'esteem3', 'esteem4', 'esteem5', 'esteem6']


VIF loop (esteem1):  33%|███▎      | 2/6 [00:01<00:02,  1.84it/s]

...

  → Dropped mom_immigrant (VIF=3.17)
  → Dropped dad_immigrant (VIF=3.06)
  → Dropped immigrant (VIF=1.50)

Winner: None
No-multicollinearity variables: ['immigrant', 'mom_immigrant', 'dad_immigrant']

======== Processing category: religious ========

🌟 Processing group with 4 variables: ['religious1', 'religious2', 'religious3', 'religious4']


  → Dropped religious1 (VIF=3.79)
  → Dropped religious2 (VIF=3.07)
  → Dropped religious4 (VIF=1.63)
  → Dropped religious3 (VIF=1.51)

Winner: None
No-multicollinearity variables: ['religious1', 'religious2', 'religious3', 'religious4']

======== Processing category: gender_attitudes ========

🌟 Processing group with 6 variables: ['ga_leaders_male_better', 'ga_edu_male_priority', 'ga_exec_male_better', 'ga_jobs_male_priority', 'ga_income_woman_problem', 'ga_women_less_corrupt']


VIF loop (ga):  17%|█▋        | 1/6 [00:00<00:01,  4.77it/s]

  → Added ga_exec_male_better (VIF=13.96)


  → Added ga_leaders_male_better (VIF=10.21)
  → Dropped ga_jobs_male_priority (VIF=7.41)
  → Dropped ga_income_woman_problem (VIF=7.00)
  → Dropped ga_edu_male_priority (VIF=6.87)
  → Dropped ga_women_less_corrupt (VIF=3.75)

Winner: ga_leaders_male_better (|corr| = 0.0058)
Survivors (high-VIF): ['ga_exec_male_better', 'ga_leaders_male_better']

Final inter-VIF winners: {'belonging': 'belonging1', 'esteem': 'esteem3', 'safety': 'safety5', 'physio': 'physio1', 'civic': 'civic53', 'trust': 'trust3', 'growth': 'growth16', 'ethical': 'ethical80', 'compassion': None, 'education': 'dad_education_level', 'socioeconomic': None, 'immigrant': None, 'religious': None, 'gender_attitudes': 'ga_leaders_male_better'}


In [ ]:
inter_grp_kept = [var for var in winner_grp_var.values() if var is not None]

non_mul_corr_grp_vars = [item for sublist in non_multi_corr_grp_vars.values() for item in sublist]
intra_vars = non_mul_corr_grp_vars + inter_grp_kept + non_grp_vars # combine the winner with ungrouped variables for intra-VIF

print(intra_vars)

['compassion1', 'compassion2', 'compassion3', 'compassion4', 'chief_wage', 'social_class', 'income_scale', 'immigrant', 'mom_immigrant', 'dad_immigrant', 'religious1', 'religious2', 'religious3', 'religious4', 'belonging1', 'esteem3', 'safety5', 'physio1', 'civic53', 'trust3', 'growth16', 'ethical80', 'dad_education_level', 'ga_leaders_male_better', 'age', 'citizenship', 'comp_good_vs_harm', 'emp_prior_immig', 'gov_vs_individual', 'income_eq_vs_diff', 'l_vs_r_political', 'live_with_parent', 'marital_status', 'num_of_child', 'num_of_household', 'pri_vs_state', 'sact', 'sex']


In [ ]:
intra_cols_kept, dropped =intra_vif(df_imputed, intra_vars, threshold=10)


🌟 Processing intra-VIF for 38 variables: ['compassion1', 'compassion2', 'compassion3', 'compassion4', 'chief_wage', 'social_class', 'income_scale', 'immigrant', 'mom_immigrant', 'dad_immigrant', 'religious1', 'religious2', 'religious3', 'religious4', 'belonging1', 'esteem3', 'safety5', 'physio1', 'civic53', 'trust3', 'growth16', 'ethical80', 'dad_education_level', 'ga_leaders_male_better', 'age', 'citizenship', 'comp_good_vs_harm', 'emp_prior_immig', 'gov_vs_individual', 'income_eq_vs_diff', 'l_vs_r_political', 'live_with_parent', 'marital_status', 'num_of_child', 'num_of_household', 'pri_vs_state', 'sact', 'sex']


Intra-VIF loop:   3%|▎         | 1/38 [00:10<06:17, 10.19s/it]

  → Dropped belonging1 (VIF=96.43)


Intra-VIF loop:   5%|▌         | 2/38 [00:18<05:26,  9.08s/it]

  → Dropped citizenship (VIF=47.83)


...

Intra-VIF loop:  32%|███▏      | 12/38 [01:24<02:39,  6.15s/it]

  → Dropped ethical80 (VIF=11.15)


Intra-VIF loop:  34%|███▍      | 13/38 [01:29<02:20,  5.64s/it]

  → Dropped l_vs_r_political (VIF=10.72)



Remaining variables after intra-VIF: ['compassion1', 'compassion2', 'compassion3', 'compassion4', 'chief_wage', 'income_scale', 'immigrant', 'mom_immigrant', 'dad_immigrant', 'religious1', 'religious2', 'religious3', 'religious4', 'dad_education_level', 'ga_leaders_male_better', 'age', 'comp_good_vs_harm', 'gov_vs_individual', 'income_eq_vs_diff', 'live_with_parent', 'marital_status', 'num_of_child', 'num_of_household', 'pri_vs_state', 'sex']


In [ ]:
final_cols = []
for c in intra_cols_kept + inter_grp_kept + ['sact']: # preserve sact
    if c not in final_cols:
        final_cols.append(c)

cat_cols = df_fea_eng.select_dtypes(include='category').columns.to_list()
final_cols = final_cols + cat_cols + [target] # happy
print(final_cols)

['compassion1', 'compassion2', 'compassion3', 'compassion4', 'chief_wage', 'income_scale', 'immigrant', 'mom_immigrant', 'dad_immigrant', 'religious1', 'religious2', 'religious3', 'religious4', 'dad_education_level', 'ga_leaders_male_better', 'age', 'comp_good_vs_harm', 'gov_vs_individual', 'income_eq_vs_diff', 'live_with_parent', 'marital_status', 'num_of_child', 'num_of_household', 'pri_vs_state', 'sex', 'belonging1', 'esteem3', 'safety5', 'physio1', 'civic53', 'trust3', 'growth16', 'ethical80', 'sact', 'society_concern', 'protect_env_vs_econ', 'immig_policy_prefer', 'freedom_eq', 'freedom_sec', 'country_aim1', 'country_aim2', 'respon_aim1', 'respon_aim2', 'important_choice1', 'important_choice2', 'employment_status', 'occupational_group', 'dad_occupational_group', 'Employment_sector', 'family_saving', 'religiosity', 'happy']


Keep only the final selected variables from VIF.

In [ ]:
df_fea_eng = df_fea_eng[final_cols]
df_fea_eng.head()

,compassion1,compassion2,compassion3,compassion4,chief_wage,income_scale,immigrant,mom_immigrant,dad_immigrant,religious1,religious2,religious3,religious4,dad_education_level,ga_leaders_male_better,age,comp_good_vs_harm,gov_vs_individual,income_eq_vs_diff,live_with_parent,marital_status,num_of_child,num_of_household,pri_vs_state,sex,belonging1,esteem3,safety5,physio1,civic53,trust3,growth16,ethical80,sact,society_concern,protect_env_vs_econ,immig_policy_prefer,freedom_eq,freedom_sec,country_aim1,country_aim2,respon_aim1,respon_aim2,important_choice1,important_choice2,employment_status,occupational_group,dad_occupational_group,Employment_sector,family_saving,religiosity,happy
0,1,0,2,2,1,3,0,0,0,<NA>,1,0,0,1,2,61,3,1,1,0,1,2,2,3,0,4,4,1,2,4,4,4,1,4,2,2,1,2,2,1,3,3,4,1,3,1,6,9,1,2,3,3
1,1,0,2,2,0,3,0,0,0,4,1,1,<NA>,1,<NA>,61,3,1,3,0,1,2,4,5,1,4,4,2,4,4,4,4,3,4,NaN,1,2,1,2,1,2,3,1,1,4,1,8,10,2,2,3,4
2,3,<NA>,<NA>,2,0,2,1,1,1,<NA>,1,1,1,4,1,42,3,1,3,0,1,4,6,3,1,4,4,2,5,3,4,4,2,5,2,NaN,2,2,2,1,2,3,4,4,1,5,0,6,NaN,2,3,3
3,2,0,2,2,0,3,0,0,0,3,1,0,<NA>,2,2,64,3,3,1,0,1,2,2,3,1,4,4,2,3,4,4,4,<NA>,3,NaN,1,2,2,1,2,1,1,2,1,4,1,8,9,2,2,3,3
4,4,2,<NA>,2,0,2,1,1,1,<NA>,1,0,0,2,<NA>,52,5,1,1,0,0,3,8,3,1,4,3,3,3,5,4,3,3,3,2,2,3,2,2,1,4,3,1,1,4,1,8,9,2,2,3,2


In [ ]:
df_fea_eng.shape

(95528, 52)

Check the variables with high missing percentage ( >15% may cause bias for nominal).

In [ ]:
na_cols_results = get_na_cols_perc(df_fea_eng,threshold=0.1)

dad_education_level : 0.14240850850012562
dad_occupational_group : 0.13609622309689307
Employment_sector : 0.2406414873126204
----------------------------------------------------------------------------------------------------
Total column:  3


#### Target Variable Cleaning
Removing rows with missing target values.

In [ ]:
# Step 1: drop NA rows found in target
target = 'happy'
df_fea_eng = df_fea_eng[~df_fea_eng[target].isna()]

df_fea_eng.shape

(94883, 52)

#### Feature Type & Missing Value Preparation
Change ordinal and numerical variables data type to support imputation

In [ ]:
df_fea_eng.dtypes

compassion1                  Int64
compassion2                  Int64
compassion3                  Int64
compassion4                  Int64
chief_wage                   Int64
income_scale                 Int64
immigrant                    Int64
mom_immigrant                Int64
dad_immigrant                Int64
religious1                   Int64
religious2                   Int64
religious3                   Int64
religious4                   Int64
dad_education_level          Int64
ga_leaders_male_better       Int64
age                          Int64
comp_good_vs_harm            Int64
gov_vs_individual            Int64
income_eq_vs_diff            Int64
live_with_parent             Int64
marital_status               Int64
num_of_child                 Int64
num_of_household             Int64
pri_vs_state                 Int64
sex                          Int64
belonging1                   Int64
esteem3                      Int64
safety5                      Int64
physio1             

In [ ]:
ord_num_cols = df_fea_eng.select_dtypes(exclude='category').columns

# convert those ordinal and numerical variables into float first so that NA can be replaced
df_fea_eng[ord_num_cols] = df_fea_eng[ord_num_cols].astype("float")

# convert into numpy nan for iterative imputing
df_fea_eng[ord_num_cols] = df_fea_eng[ord_num_cols].replace({pd.NA : np.nan})

Convert both numerical and categorical variables data from pd.NA to np.nan for imputation.

In [ ]:
# category will store NaN as <NA> implicitly
cat_cols = df_fea_eng.select_dtypes(include='category').columns

for col in cat_cols:
    # change to object to hold numpy nan
    df_fea_eng[col] = df_fea_eng[col].astype(object).where(df_fea_eng[col].notna(), np.nan)

In [ ]:
df_fea_eng.dtypes

compassion1               float64
compassion2               float64
compassion3               float64
compassion4               float64
chief_wage                float64
income_scale              float64
immigrant                 float64
mom_immigrant             float64
dad_immigrant             float64
religious1                float64
religious2                float64
religious3                float64
religious4                float64
dad_education_level       float64
ga_leaders_male_better    float64
age                       float64
comp_good_vs_harm         float64
gov_vs_individual         float64
income_eq_vs_diff         float64
live_with_parent          float64
marital_status            float64
num_of_child              float64
num_of_household          float64
pri_vs_state              float64
sex                       float64
belonging1                float64
esteem3                   float64
safety5                   float64
physio1                   float64
civic53       

Save data types.

In [ ]:
dtypes_dict = df_fea_eng.dtypes.apply(lambda x: x.name).to_dict()
with open(f'{DRIVE_DATA_PATH}/data/processed/data_types.json', 'w') as f:
    json.dump(dtypes_dict, f)

Export feature engineered data.

In [ ]:
df_fea_eng.to_csv(f'{DRIVE_DATA_PATH}/data/processed/df_fea_eng.csv', index=False)

Load feature engineered data.

In [14]:
# Load column types
with open(f'{DRIVE_DATA_PATH}/data/processed/data_types.json', 'r') as f:
    dtypes_dict = json.load(f)

# Read CSV with types
df_fea_eng = pd.read_csv(f'{DRIVE_DATA_PATH}/data/processed/df_fea_eng.csv', dtype=dtypes_dict)

#### Data Splitting

In [ ]:
target = 'happy'

X = df_fea_eng.drop(target, axis=1)
y = df_fea_eng[target]

In [ ]:
# Step 2: train/test/val split (stratified) to correct balancing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

#### Imputation Preparation

Set up imputation pipelines for categorical and numerical data.

In [ ]:
# ---------------------------
# Column groups: ordinal, nominal, discrete, continuous
# ---------------------------
continuous_cols = ['age'] # continuous
discrete_cols = [ c for c in X.columns          # discrete variables
    if X[c].nunique() >= 10 and X[c].dtype != 'object' and c not in continuous_cols]
ordinal_cols = [ c for c in X.columns           # ordinal variables
    if X[c].dropna().nunique() <= 10 and X[c].dtype != 'object'
]
nominal_cols = list(X.select_dtypes(include='object').columns) # nominal variables

print("Continuous: ",continuous_cols)
print("Discrete: ", discrete_cols)
print("Ordinal: ", ordinal_cols)
print("Nominal: ", nominal_cols)

Continuous:  ['age']
Discrete:  ['num_of_child', 'num_of_household']
Ordinal:  ['compassion1', 'compassion2', 'compassion3', 'compassion4', 'chief_wage', 'income_scale', 'immigrant', 'mom_immigrant', 'dad_immigrant', 'religious1', 'religious2', 'religious3', 'religious4', 'dad_education_level', 'ga_leaders_male_better', 'comp_good_vs_harm', 'gov_vs_individual', 'income_eq_vs_diff', 'live_with_parent', 'marital_status', 'pri_vs_state', 'sex', 'belonging1', 'esteem3', 'safety5', 'physio1', 'civic53', 'trust3', 'growth16', 'ethical80', 'sact']
Nominal:  ['society_concern', 'protect_env_vs_econ', 'immig_policy_prefer', 'freedom_eq', 'freedom_sec', 'country_aim1', 'country_aim2', 'respon_aim1', 'respon_aim2', 'important_choice1', 'important_choice2', 'employment_status', 'occupational_group', 'dad_occupational_group', 'Employment_sector', 'family_saving', 'religiosity']


Rounding and clipping functions to ensure imputed values are in valid range.

In [ ]:
def round_and_clip_array(X, columns, ranges, do_round=True):
    # Wrap as DataFrame
    df = pd.DataFrame(X, columns=columns)
    for col in columns:
        min_val, max_val = ranges[col]
        if do_round:
            df[col] = df[col].round()
        df[col] = df[col].clip(min_val, max_val) # ensure no out-of-scale data
    return df.values

def round_discrete(X, cols):
    df = pd.DataFrame(X, columns=cols)
    for col in cols:
        df[col] = df[col].round().clip(lower=0)  # ensures no negative counts
    return df.values


In [ ]:
# find the min max values for ordinal
ordinal_ranges = {
    col: (X_train[col].min(), X_train[col].max())
    for col in ordinal_cols
}

continuous_ranges = {
    col: ((X_train[col].min(), X_train[col].max()))
    for col in continuous_cols
}

# ordinal round and clip
ordinal_clip_transformer = FunctionTransformer(
    func=lambda X: round_and_clip_array(X, ordinal_cols, ordinal_ranges, do_round=True)
)

# discrete round
discrete_round_transformer = FunctionTransformer(
    func=lambda X: round_discrete(X, discrete_cols)
)

# continuous clip
continuous_clip_transformer = FunctionTransformer(
    func=lambda X: round_and_clip_array(X, continuous_cols, continuous_ranges, do_round=False)
)


Imputation pipelines.

In [ ]:
ordinal_pipeline = Pipeline([
    ("imputer", IterativeImputer(random_state=42)),
    ("round_clip", ordinal_clip_transformer)
])

continuous_pipeline = Pipeline([
    ("imputer", IterativeImputer(random_state=42)),
    ("clip", continuous_clip_transformer),
    ("scaler", StandardScaler())
])

discrete_pipeline = Pipeline([
    ("imputer", IterativeImputer(random_state=42)),
    ("round_discrete", discrete_round_transformer)
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("ordinal", ordinal_pipeline, ordinal_cols),
    ("continuous", continuous_pipeline, continuous_cols),
    ("discrete", discrete_pipeline, discrete_cols),
    ("nominal", nominal_pipeline, nominal_cols)
])


#### Label Encode Target

Ensure the target variable (happy) starts from 0 for easier readiness in further evaluation.

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_train = pd.DataFrame(y_train, columns=["happy"])

y_test = label_encoder.transform(y_test)
y_test = pd.DataFrame(y_test, columns=["happy"])

y_val = label_encoder.transform(y_val)
y_val = pd.DataFrame(y_val, columns=["happy"])

### RFECV

In [ ]:
# IMPORTANT: enable pandas output so RFECV receives column names
preprocessor.set_output(transform="pandas")

ColumnTransformer(transformers=[('ordinal',
                                 Pipeline(steps=[('imputer',
                                                  IterativeImputer(random_state=42)),
                                                 ('round_clip',
                                                  FunctionTransformer(func=<function <lambda> at 0x7e11d8e27240>))]),
                                 ['compassion1', 'compassion2', 'compassion3',
                                  'compassion4', 'chief_wage', 'income_scale',
                                  'immigrant', 'mom_immigrant', 'dad_immigrant',
                                  'religious1', 'religious2', 'religious3'...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['society_concern', 'protect_env_vs_econ',
                                  'immig_policy_prefer', 'freedom_eq',
                                  'freedom_sec', 'country_aim1', 'country_aim2',
                                  'respon_aim1', 'respon_aim2',
                                  'important_choice1', 'important_choice2',
                                  'employment_status', 'occupational_group',
                                  'dad_occupational_group', 'Employment_sector',
                                  'family_saving', 'religiosity'])])

Run RFECV with the preprocessor using pipeline to avoid data leakage.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV

rfecv = RFECV(
    estimator=RandomForestClassifier(random_state=42),
    step=1,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rfecv_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", rfecv)
])


In [ ]:
y_train_1d = y_train.values.ravel() # use ravel to avoid DataConversionWarning
rfecv_pipeline.fit(X_train, y_train_1d)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('ordinal',
                                                  Pipeline(steps=[('imputer',
                                                                   IterativeImputer(random_state=42)),
                                                                  ('round_clip',
                                                                   FunctionTransformer(func=<function <lambda> at 0x7e11d8e27240>))]),
                                                  ['compassion1', 'compassion2',
                                                   'compassion3', 'compassion4',
                                                   'chief_wage', 'income_scale',
                                                   'immigrant', 'mom_immigrant',
                                                   'dad_immigrant',
                                                   'relig...
                                                   'freedom_eq', 'freedom_sec',
                                                   'country_aim1',
                                                   'country_aim2',
                                                   'respon_aim1', 'respon_aim2',
                                                   'important_choice1',
                                                   'important_choice2',
                                                   'employment_status',
                                                   'occupational_group',
                                                   'dad_occupational_group',
                                                   'Employment_sector',
                                                   'family_saving',
                                                   'religiosity'])])),
                ('selector',
                 RFECV(cv=5, estimator=RandomForestClassifier(random_state=42),
                       n_jobs=-1, scoring='accuracy'))])

In [ ]:
import numpy as np

selector = rfecv_pipeline.named_steps["selector"]
pre = rfecv_pipeline.named_steps["preprocessor"]

all_feature_names = []

for name, trans, cols in pre.transformers_:
    # Case 1: transformer is a Pipeline
    if hasattr(trans, "named_steps"):
        last_step = trans.named_steps[next(reversed(trans.named_steps))]
        if hasattr(last_step, "get_feature_names_out"):
            names = last_step.get_feature_names_out(cols)
        else:
            names = cols
    # Case 2: direct transformer with names (e.g., OneHotEncoder)
    elif hasattr(trans, "get_feature_names_out"):
        names = trans.get_feature_names_out(cols)
    # Case 3: no feature name support → fallback to original column names
    else:
        names = cols

    all_feature_names.extend(names)

all_feature_names = np.array(all_feature_names)

# Now apply RFECV mask
selected_features = all_feature_names[selector.support_]

print("Optimal number of features:", selector.n_features_)
print("Selected features:", list(selected_features))


Optimal number of features: 69
Selected features: [np.str_('compassion1'), np.str_('compassion2'), np.str_('compassion3'), np.str_('compassion4'), np.str_('chief_wage'), np.str_('income_scale'), np.str_('religious1'), np.str_('religious2'), np.str_('religious3'), np.str_('religious4'), np.str_('dad_education_level'), np.str_('ga_leaders_male_better'), np.str_('comp_good_vs_harm'), np.str_('gov_vs_individual'), np.str_('income_eq_vs_diff'), np.str_('live_with_parent'), np.str_('marital_status'), np.str_('pri_vs_state'), np.str_('sex'), np.str_('esteem3'), np.str_('safety5'), np.str_('physio1'), np.str_('civic53'), np.str_('trust3'), np.str_('growth16'), np.str_('ethical80'), np.str_('sact'), np.str_('age'), np.str_('num_of_child'), np.str_('num_of_household'), np.str_('society_concern_1'), np.str_('society_concern_2'), np.str_('protect_env_vs_econ_1'), np.str_('protect_env_vs_econ_2'), np.str_('immig_policy_prefer_2'), np.str_('immig_policy_prefer_3'), np.str_('freedom_eq_1'), np.str_('

In [ ]:
# Transform the original DataFrames
X_train_transformed = rfecv_pipeline.named_steps["preprocessor"].transform(X_train)
X_val_transformed   = rfecv_pipeline.named_steps["preprocessor"].transform(X_val)
X_test_transformed  = rfecv_pipeline.named_steps["preprocessor"].transform(X_test)

# Ensure the transformed outputs are NumPy arrays
X_train_transformed = np.array(X_train_transformed)
X_val_transformed   = np.array(X_val_transformed)
X_test_transformed  = np.array(X_test_transformed)

In [ ]:
# selector is RFECV selector
selector = rfecv_pipeline.named_steps["selector"]

# convert to dataframe
X_train_final = pd.DataFrame(
    X_train_transformed[:, selector.support_],
    columns=selected_features
)
X_val_final = pd.DataFrame(
    X_val_transformed[:, selector.support_],
    columns=selected_features
)
X_test_final = pd.DataFrame(
    X_test_transformed[:, selector.support_],
    columns=selected_features
)

In [ ]:
# Retrieve columns which are filtered by RFECV
excluded_mask = ~selector.support_

# convert to dataframe
X_train_exc = pd.DataFrame(
    X_train_transformed[:, excluded_mask]
)
X_test_exc = pd.DataFrame(
    X_test_transformed[:, excluded_mask]
)
X_val_exc = pd.DataFrame(
    X_val_transformed[:, excluded_mask]
)


excluded_features = np.array(all_feature_names)[excluded_mask]

X_train_exc.columns = excluded_features
X_val_exc.columns = excluded_features
X_test_exc.columns = excluded_features


In [ ]:
# Export final X train ,test and validation sets with the belonging1

X_train_final = pd.concat(
    [X_train_exc['belonging1'], X_train_final],
    axis=1
)

X_test_final = pd.concat(
    [X_test_exc['belonging1'], X_test_final],
    axis=1
)

X_val_final = pd.concat(
    [X_val_exc['belonging1'], X_val_final],
    axis=1
)

X_train_final.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/X_train_final_ori.csv', index=False)
X_test_final.to_csv(f'{DRIVE_DATA_PATH}/data/test_sets/X_test_final.csv', index=False)
X_val_final.to_csv(f'{DRIVE_DATA_PATH}/data/val_sets/X_val_final.csv', index=False)

In [15]:
X_train_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/train_sets/X_train_final_ori.csv')
X_test_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/test_sets/X_test_final.csv')
X_val_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/val_sets/X_val_final.csv')

#### Target Class Granularity Selection

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# --- Step 1: Define binning schemes (hierarchical, from 0-based) ---
binning_steps = {
    '4-class': {0:0, 1:1, 2:2, 3:3},  # original 4 classes
    '3-class': {0:0, 1:1, 2:2, 3:2},  # merge top two classes
    '2-class': {0:0, 1:0, 2:1, 3:1}   # binary: unhappy vs happy
}

# --- Step 2: Evaluation function ---
def evaluate_model(X_train, y_train, X_val, y_val, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

# --- Step 3: Evaluate sequentially and update labels ---
results = []
y_train_versions = {}
y_val_versions = {}
y_test_versions = {}

# Start with original y_train, y_val
y_train_base = y_train["happy"].copy()
y_val_base = y_val["happy"].copy()
y_test_base = y_test["happy"].copy()

for scheme_name in ['4-class', '3-class', '2-class']:
    bin_map = binning_steps[scheme_name]

    # Map train and val labels based on current labels (which update each iteration)
    y_train_binned = y_train_base.map(bin_map).astype(int)
    y_val_binned = y_val_base.map(bin_map).astype(int)
    y_test_binned = y_test_base.map(bin_map).astype(int)

    # Train and evaluate model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    acc_val = evaluate_model(X_train_final, y_train_binned, X_val_final, y_val_binned, model)
    acc_test = evaluate_model(X_train_final, y_train_binned, X_test_final, y_test_binned, model)

    # Store results
    results.append({
        'Scheme': scheme_name,
        'Num_Classes': len(set(bin_map.values())),
        'Validation Accuracy': acc_val,
        'Test Accuracy': acc_test
    })

    # Store the binned versions after evaluation
    y_train_versions[scheme_name] = y_train_binned
    y_val_versions[scheme_name] = y_val_binned
    y_test_versions[scheme_name] = y_test_binned

    # Update base labels for next iteration
    y_train_base = y_train_binned
    y_val_base = y_val_binned
    y_test_base = y_test_binned

# Final: show sorted results
results_df = pd.DataFrame(results).sort_values('Validation Accuracy', ascending=False)
print(results_df)

    Scheme  Num_Classes  Validation Accuracy  Test Accuracy
2  2-class            2             0.876335       0.874868
1  3-class            3             0.867552       0.865454
0  4-class            4             0.639896       0.641959


In [ ]:
y_train_final = y_train_versions['2-class']
y_val_final = y_val_versions['2-class']
y_test_final = y_test_versions['2-class']

In [ ]:
# Export final y train ,test and validation sets
y_train_final.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/y_train_final_ori.csv', index=False)
y_test_final.to_csv(f'{DRIVE_DATA_PATH}/data/test_sets/y_test_final.csv', index=False)
y_val_final.to_csv(f'{DRIVE_DATA_PATH}/data/val_sets/y_val_final.csv', index=False)

In [16]:
y_train_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/train_sets/y_train_final_ori.csv')
y_test_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/test_sets/y_test_final.csv')
y_val_final = pd.read_csv(f'{DRIVE_DATA_PATH}/data/val_sets/y_val_final.csv')

#### SMOTE

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def run_imbalance_experiment(
    sampler,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    model=None,
    name="method"
):
    """
    sampler: imblearn sampler object OR None
    """

    # Default model
    if model is None:
        model = RandomForestClassifier(random_state=42)

    # =========================
    # 1. Resampling step
    # =========================
    if sampler is None:
        X_res, y_res = X_train, y_train
    else:
        X_res, y_res = sampler.fit_resample(X_train, y_train)

    # =========================
    # 2. Train
    # =========================
    model.fit(X_res, y_res)

    # =========================
    # 3. Predict
    # =========================
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    results = {
        "method": name,
        "X_resampled": X_res,
        "y_resampled": y_res,
        "y_val_pred": y_val_pred,
        "y_test_pred": y_test_pred,
        "val_accuracy": accuracy_score(y_val, y_val_pred),
        "test_accuracy": accuracy_score(y_test, y_test_pred)
    }

    print(f"\n=== {name} ===")
    print(f"Validation Accuracy: {results['val_accuracy']:.6f}")
    print(f"Test Accuracy: {results['test_accuracy']:.6f}")

    return results

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

samplers = [
    (None, "No Resampling"),
    (SMOTE(random_state=42), "SMOTE"),
    (ADASYN(random_state=42), "ADASYN"),
    (RandomUnderSampler(random_state=42), "Undersampling")
]

results = []

for sampler, name in samplers:
    res = run_imbalance_experiment(
        sampler=sampler,
        X_train=X_train_final,
        y_train=y_train_final,
        X_val=X_val_final,
        y_val=y_val_final,
        X_test=X_test_final,
        y_test=y_test_final,
        name=name
    )
    results.append(res)


=== No Resampling ===
Validation Accuracy: 0.876335
Test Accuracy: 0.874868

=== SMOTE ===
Validation Accuracy: 0.876967
Test Accuracy: 0.876695

=== ADASYN ===
Validation Accuracy: 0.878021
Test Accuracy: 0.877889

=== Undersampling ===
Validation Accuracy: 0.759556
Test Accuracy: 0.752196


In [ ]:
from sklearn.metrics import classification_report

for res in results:
    print(f"\n=== {res['method']} ===")
    print(classification_report(y_test_final, res['y_test_pred'], digits=4))


=== No Resampling ===
              precision    recall  f1-score   support

           0     0.7048    0.2229    0.3387      2046
           1     0.8830    0.9843    0.9309     12187

    accuracy                         0.8749     14233
   macro avg     0.7939    0.6036    0.6348     14233
weighted avg     0.8574    0.8749    0.8458     14233


=== SMOTE ===
              precision    recall  f1-score   support

           0     0.6694    0.2810    0.3959      2046
           1     0.8900    0.9767    0.9313     12187

    accuracy                         0.8767     14233
   macro avg     0.7797    0.6289    0.6636     14233
weighted avg     0.8583    0.8767    0.8544     14233


=== ADASYN ===
              precision    recall  f1-score   support

           0     0.6906    0.2727    0.3910      2046
           1     0.8892    0.9795    0.9321     12187

    accuracy                         0.8779     14233
   macro avg     0.7899    0.6261    0.6616     14233
weighted avg     0.8

Pick with SMOTE

In [ ]:
res_smote = results[1]
print(res_smote['method'])

X_train_final_smote = res_smote["X_resampled"]
y_train_final_smote = res_smote["y_resampled"]

X_train_final_smote.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/X_train_final_smote.csv', index=False)
y_train_final_smote.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/y_train_final_smote.csv', index=False)

SMOTE


In [ ]:
# res_adasyn = results[2]
# print(res_adasyn['method'])

# X_train_final_adasyn = res_adasyn["X_resampled"]
# y_train_final_adasyn = res_adasyn["y_resampled"]

# X_train_final_adasyn.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/X_train_final_adasyn.csv', index=False)
# y_train_final_adasyn.to_csv(f'{DRIVE_DATA_PATH}/data/train_sets/y_train_final_adasyn.csv', index=False)

ADASYN
